In [1]:
!pip3 install -q wget
!pip3 install -q sentence-transformers
!pip install -q faiss-cpu

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 57.8 MB/s eta 0:00:00


In [2]:
import os
import wget
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [3]:
cvs_file_path = 'https://raw.githubusercontent.com/openai/openai-cookbook/main/examples/data/AG_news_samples.csv'
file_path = 'AG_news_samples.csv'

if not os.path.exists(file_path):
    wget.download(cvs_file_path, file_path)
    print('File downloaded successfully.')
else:
    print('File already exists in the local file system.')

df = pd.read_csv('AG_news_samples.csv')
df

File downloaded successfully.


,title,description,label_int,label
0,World Briefings,BRITAIN: BLAIR WARNS OF CLIMATE THREAT Prime M...,1,World
1,Nvidia Puts a Firewall on a Motherboard (PC Wo...,PC World - Upcoming chip set will include buil...,4,Sci/Tech
2,"Olympic joy in Greek, Chinese press",Newspapers in Greece reflect a mixture of exhi...,2,Sports
3,U2 Can iPod with Pictures,"SAN JOSE, Calif. -- Apple Computer (Quote, Cha...",4,Sci/Tech
4,The Dream Factory,"Any product, any shape, any size -- manufactur...",4,Sci/Tech
...,...,...,...,...
1995,You Control: iTunes puts control in OS X menu ...,MacCentral - You Software Inc. announced on Tu...,4,Sci/Tech
1996,Argentina beat Italy for place in football final,Favourites Argentina beat Italy 3-0 this morni...,2,Sports
1997,NCAA case no worry for Spurrier,Shortly after Steve Spurrier arrived at Florid...,2,Sports
1998,Secret Service Busts Cyber Gangs,The US Secret Service Thursday announced arres...,4,Sci/Tech


In [4]:
# Convert to list of records
data = df.to_dict(orient="records")

# Load embedding model
model = SentenceTransformer("flax-sentence-embeddings/all_datasets_v3_mpnet-base")

# Extract descriptions and embed
descriptions = [item['description'] for item in data]
embeddings = model.encode(descriptions, convert_to_numpy=True)
embeddings = embeddings.astype("float32")  # FAISS requires float32

print("Embedding shape:", embeddings.shape)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.85k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (2000, 768)


In [5]:
embedding_dim = embeddings.shape[1]

# Using L2 (Euclidean) distance — use IndexFlatIP for cosine-like behavior
index = faiss.IndexFlatL2(embedding_dim)

# Add all vectors to the index
index.add(embeddings)

print(f"Total vectors in index: {index.ntotal}")

Total vectors in index: 2000


In [6]:
# Create a query
query_text = "Government increases interest rates"
query_vector = model.encode([query_text], convert_to_numpy=True).astype("float32")

# Search top 3 most similar documents
k = 3
distances, indices = index.search(query_vector, k)

# Show results
for i, idx in enumerate(indices[0]):
    result = data[idx]
    print(f"\n Result {i+1}")
    print(f"Distance Score: {distances[0][i]:.4f}")
    print(f"Title: {result['title']}")
    print(f"Description: {result['description']}")


 Result 1
Distance Score: 0.8672
Title: Rates rise on T-bills
Description: Interest rates on short-term Treasury securities rose in yesterday's auction. The Treasury Department sold \$19 billion in three-month bills at a discount rate of 1.710 percent, up from 1.685 percent last week. An additional \$17 billion was sold in six-month bills at a rate of 1.950 percent, up from 1.870 percent.

 Result 2
Distance Score: 0.8704
Title: Federal Reserve Raises Benchmark A 5th Time
Description: Federal Reserve officials raised a key short-term interest rate Tuesday for the fifth time this year, and indicated they will gradually move rates higher next year to keep inflation under control as the economy expands.

 Result 3
Distance Score: 0.9397
Title: Mortgage rates at Canadian banks headed higher after Bank of &lt;b&gt;...&lt;/b&gt;
Description: TORONTO (CP) - Canada #39;s big banks are increasing mortgage rates following a decision by the Bank of Canada to raise its overnight rate by one-quart

In [7]:
# Saving and Loading

# Save the index to disk
# faiss.write_index(index, "faiss_agnews.index")

# Load it later
# index = faiss.read_index("faiss_agnews.index")